# AeroPV-Trust: Multimodal UAV Photovoltaic Anomaly Inspection & Trustworthy XAI
**Author:** Labonna Prava  
**Project:** End-to-end framework for aerial solar module segmentation, dual-branch late fusion deep learning, and multi-level explainability (Grad-CAM, Modality Ablation, Prototype Retrieval, CCME).

---

In [ ]:
# 1. Environment Setup and Project Imports
import os
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

# Ensure project root is available
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, CROPS_DIR, CHECKPOINTS_DIR, RESULTS_DIR, DEVICE
from src.data.thermogram import Thermogram, load_rgb, load_thermal
from src.data.segmentation import make_aligned_pair, extract_module_crops, crop_image
from src.data.dataset import create_dataloaders
from src.models.late_fusion import LateFusion
from src.training.trainer import evaluate_model
from src.explainability.gradcam import GradCAM, generate_cam_overlay
from src.explainability.ablation import evaluate_modality_ablation
from src.explainability.prototypes import PrototypeRetriever
from src.explainability.counterfactual import CrossModalCounterfactualExplainer

print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

## 2. Radiometric Thermal Processing & Alignment
Raw UAV thermal captures are stored as 16-bit radiometric TIFFs. We calibrate to Celsius using:
$$\text{Celsius} = (\text{raw} \times 0.04) - 273.15$$
We then align the visual RGB image with the thermal field of view.

In [ ]:
# Load sample image pair
rgb_dir = DATA_DIR / "double-row" / "rgb images"
th_dir = DATA_DIR / "double-row" / "thermal images"

sample_rgb = sorted(list(rgb_dir.glob("*.jpg")))[0]
sample_th = th_dir / f"{sample_rgb.stem.replace('_RGB', '_T')}.tif"

rgb_raw = load_rgb(sample_rgb)
th_obj = load_thermal(sample_th)
th8_raw = th_obj.render8()

rgb_aligned, th8_aligned = make_aligned_pair(rgb_raw, th8_raw)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(rgb_aligned, cv2.COLOR_BGR2RGB))
axes[0].set_title("Aligned RGB Inspection Region")
axes[0].axis("off")

axes[1].imshow(th8_aligned, cmap="inferno")
axes[1].set_title("Radiometric Thermal IR (Inferno)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 3. Computer Vision Module Segmentation & Extraction
We extract PV modules via multi-channel HSV Otsu binarization, morphological hole filling, and connected-component labeling. Each module is assigned a robust Median Absolute Deviation (MAD) thermal anomaly score:
$$\text{Score} = \frac{T_{95}(\text{module}) - \text{median}(\text{scene})}{\text{MAD}(\text{scene})}$$

In [ ]:
from src.config import IR_CROP
thC_crop = crop_image(th_obj.celsius, IR_CROP)
samples, mask01, labels = extract_module_crops(rgb_aligned, th8_aligned, thC_crop)

print(f"Detected {len(samples)} PV modules in the scan region.")
if samples:
    print(f"Sample Module 0: Anomaly Score = {samples[0]['score']:.3f}, BBox = {samples[0]['bbox']}")

## 4. Multimodal Late-Fusion Anomaly Classification
We evaluate our dual-branch late fusion architecture combining modality-specific ResNet encoders with a shared 512-d classification head.

In [ ]:
index_csv = CROPS_DIR / "index.csv"
_, test_loader, _, test_df = create_dataloaders(index_csv=index_csv, batch_size=16)

model = LateFusion(num_classes=2).to(DEVICE)
ckpt = CHECKPOINTS_DIR / "late_fusion.pt"
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print("Loaded trained Late Fusion checkpoint successfully.")

metrics = evaluate_model(model, test_loader, device=DEVICE)
print("Test Set Performance Metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:15s}: {v:.4f}")

## 5. Multi-Level Explainability Suite (XAI)
### A. Spatial Attribution with Grad-CAM
Visualizing where the convolutional representations concentrate attention for anomalous vs. normal modules.

In [ ]:
sample_rgb_t, sample_th_t, sample_y = next(iter(test_loader))
sample_rgb_t = sample_rgb_t[0:1].to(DEVICE)
sample_th_t = sample_th_t[0:1].to(DEVICE)

rgb_layer = model.rgb_net.layer4[-1].conv2
with GradCAM(model, rgb_layer) as gc:
    cam_rgb, pred_cls, prob = gc(lambda: model(sample_rgb_t, sample_th_t), class_idx=1)

print(f"Grad-CAM generated for predicted class {pred_cls} with probability {prob:.4f}")

### B. Modality Ablation Analysis
Quantifying the marginal utility of RGB vs. Thermal modalities across the test population.

In [ ]:
ablation = evaluate_modality_ablation(model, test_loader, device=DEVICE)
for k, v in ablation.items():
    print(f"{k}: {v:.4f}")

### C. Prototype Retrieval & Counterfactual CCME
Retrieving nearest-neighbor training cases and computing latent transition factors ($\alpha$).

In [ ]:
retriever = PrototypeRetriever(model, device=DEVICE)
retriever.build_index(test_loader)
matches = retriever.query(sample_rgb_t[0], sample_th_t[0], top_k=3)

print("Top Retrieved Training Exemplars:")
for m in matches:
    print(f"  Rank {m['rank']}: Cosine Similarity = {m['similarity']:.4f}, Label = {m['label']}")

ccme = CrossModalCounterfactualExplainer(model, device=DEVICE)
ccme.fit_normal_baseline(test_loader)
cf = ccme.explain(sample_rgb_t[0], sample_th_t[0])
print("\nCross-Modal Counterfactual Boundary:")
for k, v in cf.items():
    print(f"  {k}: {v}")